<a href="https://colab.research.google.com/github/mkhanamm/Information-Retrieval-System/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

tokenization, normalization, stopword removal, stemming

**Part 1: Text Preprocessing Pipeline for Hindi (Devanagari Script)**

In [8]:
import re
import unicodedata
from pathlib import Path

# **Normalization**

In [17]:
def normalize_text(text: str) -> str:
#Apply Unicode NFC normalization
  text =unicodedata.normalize("NFC", text)

#Remove zero-width joiners
  text = text.replace("\u200D", "") #zero width joiner : Join characters together
  text = text.replace("\u200C", "") #zero width non joiner: Prevent characters from joining

 #Many writing systems (Arabic, Persian, Urdu, some Indic scripts) change shape depending on neighboring letters, hence its needed

  # Normalize nasalization variants
  text = text.replace("ँ", "ं")
  # Nukhta normalization
  text = text.replace("क़", "क")
  text = text.replace("ख़", "ख")
  text = text.replace("ग़", "ग")
  text = text.replace("ज़", "ज")
  text = text.replace("फ़", "फ")
  text = text.replace("ड़", "ड")
  text = text.replace("ढ़", "ढ")

   # matra (vowel sign) long → short
  text = text.replace("ी", "ि")
  text = text.replace("ू", "ु")
  # independent vowels
  text = text.replace("ई", "इ")
  text = text.replace("ऊ", "उ")

  return text


Full normalization pipeline.

    Parameters
    ----------
    text                  : raw text
    remove_joiners        : remove ZWJ/ZWNJ
    fix_diacritics        : apply diacritic/nukta normalization
    collapse_vowel_length : collapse long vowels → short (optional, lossy)

# **Tokenization**

In [10]:
_DEVA_WORD_RE = re.compile(r"[\u0900-\u097F\u0966-\u096F\w]+")
#hindi word pattern: \u0900-\u097F: hindi charachters; \u0966-\u0  96F: hindi digits; \w:Eng letters, numb..; +: matches single ch as one token
_HINDI_PUNCT_RE = re.compile( r"[।॥,!?\"'()\[\]{}<>:;@#\$%^&*+=|\\\/~`।॥\u0964\u0965]")
#Hindi-specific punctuation / numeral patterns to strip before tokenization
_DEVA_DIGIT_MAP = str.maketrans("०१२३४५६७८९", "0123456789")
#Devanagari digit map  ०१२३…९  →  0-9

def tokenize(text: str, normalize_digits: bool = True) -> list[str]:
 # Remove punctuation (keeps Devanagari letters, matras, digits)
  cleaned = _HINDI_PUNCT_RE.sub(" ", text)
  tokens = _DEVA_WORD_RE.findall(cleaned)
  if normalize_digits: #if true then map to ascii, else remain hindi no.
        tokens = [t.translate(_DEVA_DIGIT_MAP) for t in tokens]
  return [t for t in tokens if t.strip()]

# Steps:
#       1. Strip punctuation / dandas (। ॥)
#       2. Extract Devanagari + Latin tokens via regex
#       3. Optionally map Devanagari digits to ASCII

re.sub() : Find something, Replace it with something else

re.findall() : Search and give me all matches; result = re.findall(r"\w+", text)

re.complie() :word_pattern = re.compile(r"\w+"); finding \w pattern nd saving that pattern.

Now:result = word_pattern.findall(text) finds that w pattern in any text .(basically reusing)




# **Stopword** **removal**

In [11]:
def load_stopwords(filepath: str = "stopwords_hi.txt") -> set[str]:

    path = Path(filepath)
    if not path.exists():
        # fallback minimal list
        return {
            "और", "है", "में", "के", "की", "का", "को", "से",
            "पर", "यह", "वह", "एक", "हैं", "कि", "भी", "था"
        }
    words = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            words.add(normalize_text(line))
    return words

def remove_stopwords(tokens: list[str], stopwords: set[str]) -> list[str]:
   return [t for t in tokens if t not in stopwords]
#returns filtered token list.

# **Rule Based Stemmer**

 Rule-based Hindi stemmer.

    Strips the longest matching suffix from a word.


In [37]:
HINDI_SUFFIXES = [
    # Long nominal plural suffixes (must come before shorter ones)
    "ियाँ", "ियां", "ियों",
    "ाओं", "ाएँ", "ाएं",
    # Verbal compound suffixes
    "ताओ", "ताएँ", "ताए",
    "ेगा", "ेगी", "ेगे",
    "ोगे", "ोगी", "ोगा",
    "ाना", "ाकर",
    "ती", "ता", "ते",
    "ना", "नी", "ने",
    "या", "यी", "ये",
    # Nominal suffixes
    "ओं", "ों",
    "ान", "आन",
    "ार", "आर",
    "ाई", "ाइ",
    "पन",
    "ें",
    # Short suffixes last
    "ाँ", "ाओ",
    "ो", "ी", "े", "ा",
]
def stem(word: str) -> str:
 # Avoid stemming very short words
 if len(word) <= 2:
   return word
 for SUFFIX in HINDI_SUFFIXES:
  if word.endswith(SUFFIX):
    stemmed = word[: -len(SUFFIX)]
    # Ensure minimum stem length
    if len(stemmed) >= 2:
               return stemmed

 return word

def batch_stem(words: list[str]) -> list[tuple[str, str]]:
    """Return list of (original, stemmed) pairs."""
    return [(w, stem(w)) for w in words]



# **Full Pipeline**

In [41]:
def preprocess(text: str,
               stopwords: set[str] | None = None,
               apply_stemming: bool = False,
               ) -> list[str]:

    text = normalize_text(text)
    tokens = tokenize(text)
    if stopwords:
        tokens = remove_stopwords(tokens, stopwords)
    if apply_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

# **Running Test**

1. NORMALIZATION TESTS

In [22]:
print("Testing normalization...")
text = "फ़रीदा ने क़िले में झूला झूला, चाँदनी रात में भारत\u200D के १२३ विद्यार्थी गीत गा रहे थे।"
result = normalize_text(text)
print("Input :", text)
print("Output:", result)
print()



Testing normalization...
Input : फ़रीदा ने क़िले में झूला झूला, चाँदनी रात में भारत‍ के १२३ विद्यार्थी गीत गा रहे थे।
Output: फ़रिदा ने क़िले में झुला झुला, चांदनि रात में भारत के १२३ विद्यार्थि गित गा रहे थे।



2.TOCKENIZATION TEST

In [28]:
print("Testing tokenization")
text = "भारत में १२३ विद्यार्थी Python सीख रहे हैं। वर्ष 2025 में AI और NLP बहुत लोकप्रिय हैं!"
tokens = tokenize(text)
print("Tokens:", tokens)
token2 = tokenize("मेरे पास ५ किताबें।", normalize_digits=False)
print("No normalization of digits when flag is off:")
print(token2)

Testing tokenization
Tokens: ['भारत', 'में', '123', 'विद्यार्थी', 'Python', 'सीख', 'रहे', 'हैं', 'वर्ष', '2025', 'में', 'AI', 'और', 'NLP', 'बहुत', 'लोकप्रिय', 'हैं']
No normalization of digits when flag is off:
['मेरे', 'पास', '५', 'किताबें']


3.STOPWORD REMOVAL TESTS

In [27]:
tokens_in   = ["राम", "और", "सीता", "वन", "में", "गए"]
stopwords = load_stopwords()
filtered = remove_stopwords(tokens_in, stopwords)
print(filtered)

['राम', 'सीता', 'वन', 'गए']


4. STEM TESTING

In [38]:
print("Testing stemmer")
words = ["लड़कों", "लड़कियाँ", "खेलता", "घर"]
result = batch_stem(words)
print(result)

Testing stemmer
[('लड़कों', 'लड़क'), ('लड़कियाँ', 'लड़क'), ('खेलता', 'खेल'), ('घर', 'घर')]


5.Pipeline Test

In [42]:
print("Testing Full Pipeline")

text = "फ़रीदा ने क़िले में १२३ विद्यार्थियों के साथ खेलता हुआ गीत गाया और चाँदनी रात का आनंद लिया।"

stopwords = load_stopwords()

result = preprocess(
    text,
    stopwords=stopwords,
    apply_stemming=True,

)

print(result)

Testing Full Pipeline
['फ़रिद', 'ने', 'क़िल', '123', 'विद्यार्थ', 'साथ', 'खेल', 'हुआ', 'गित', 'गा', 'चांदनि', 'रात', 'आनंद', 'लि']
